# Imports

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append(str(Path("..").resolve()))
from src.utils import DATA_RAW, DATA_PROC, get_logger

logger = get_logger("feature_engineering")

# Load Data

In [ ]:
df = pd.read_csv(DATA_RAW / "application_train.csv")
df_test = pd.read_csv(DATA_RAW / "application_test.csv")

logger.info(f"Train: {df.shape} | Test: {df_test.shape}")

# Separate target before any processing
y = df["TARGET"].copy()
df = df.drop(columns=["TARGET"])

# Track SK_ID_CURR for joining later
train_ids = df["SK_ID_CURR"].copy()
test_ids  = df_test["SK_ID_CURR"].copy()

# Combine for consistent transformations
df["_split"] = "train"
df_test["_split"] = "test"
combined = pd.concat([df, df_test], axis=0, ignore_index=True)
logger.info(f"Combined shape: {combined.shape}")

# Drop extreme missing columns

In [ ]:
# Columns above 60% missing — not worth imputing
missing_pct = combined.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.60].index.tolist()
logger.info(f"Dropping {len(cols_to_drop)} columns with >60% missing")

combined.drop(columns=cols_to_drop, inplace=True)

# Save dropped list for reproducibility
with open(DATA_PROC / "dropped_cols.json", "w") as f:
    json.dump(cols_to_drop, f, indent=2)

# Missingness indicator flags

In [ ]:
# For columns 40–60% missing, missingness itself is a signal
missing_pct_remaining = combined.isnull().mean()
flag_cols = missing_pct_remaining[
    (missing_pct_remaining > 0.40) & (missing_pct_remaining <= 0.60)
].index.tolist()

for col in flag_cols:
    combined[f"{col}_MISSING"] = combined[col].isnull().astype(np.int8)

logger.info(f"Created {len(flag_cols)} missingness indicator flags")

# Fix known data anomalies

In [ ]:
# DAYS_EMPLOYED has 365243 as a placeholder for "not employed" — a known quirk
combined["DAYS_EMPLOYED_ANOMALY"] = (combined["DAYS_EMPLOYED"] == 365243).astype(np.int8)
combined["DAYS_EMPLOYED"] = combined["DAYS_EMPLOYED"].replace(365243, np.nan)

# Convert negative day columns to positive (they represent days before application)
day_cols = [c for c in combined.columns if "DAYS_" in c]
for col in day_cols:
    if combined[col].min() < 0:
        combined[col] = combined[col].abs()

logger.info(f"Fixed anomalies in DAYS_EMPLOYED, converted {len(day_cols)} day columns to positive")

# Domain-specific ratio features

In [ ]:
# These ratios encode financial stress signals — LightGBM can't discover these on its own
combined["CREDIT_INCOME_RATIO"]    = combined["AMT_CREDIT"]   / (combined["AMT_INCOME_TOTAL"] + 1)
combined["ANNUITY_INCOME_RATIO"]   = combined["AMT_ANNUITY"]  / (combined["AMT_INCOME_TOTAL"] + 1)
combined["CREDIT_TERM"]            = combined["AMT_ANNUITY"]  / (combined["AMT_CREDIT"] + 1)
combined["GOODS_CREDIT_RATIO"]     = combined["AMT_GOODS_PRICE"] / (combined["AMT_CREDIT"] + 1)
combined["INCOME_PER_PERSON"]      = combined["AMT_INCOME_TOTAL"] / (combined["CNT_FAM_MEMBERS"] + 1)
combined["CHILDREN_RATIO"]         = combined["CNT_CHILDREN"]  / (combined["CNT_FAM_MEMBERS"] + 1)

# Age in years (more interpretable for SHAP later)
combined["AGE_YEARS"]              = combined["DAYS_BIRTH"] / 365

# Employment stability ratio
combined["EMPLOYMENT_STABILITY"]   = combined["DAYS_EMPLOYED"] / (combined["DAYS_BIRTH"] + 1)

logger.info("Created 8 domain ratio features")

# EXT_SOURCE interaction features

In [ ]:
# EXT_SOURCE 1/2/3 are external credit scores — the strongest predictors
# Their interactions capture multi-bureau agreement/disagreement

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

combined["EXT_SOURCE_MEAN"]   = combined[ext_cols].mean(axis=1)
combined["EXT_SOURCE_STD"]    = combined[ext_cols].std(axis=1)
combined["EXT_SOURCE_MIN"]    = combined[ext_cols].min(axis=1)
combined["EXT_SOURCE_PROD"]   = combined[ext_cols].prod(axis=1)

# High std = bureaus disagree = higher uncertainty = higher risk
combined["EXT_SOURCE_DISAGREEMENT"] = combined["EXT_SOURCE_STD"] / (combined["EXT_SOURCE_MEAN"] + 1e-5)

logger.info("Created 5 EXT_SOURCE interaction features")

# Encode categoricals

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = combined.select_dtypes("object").columns.tolist()
cat_cols = [c for c in cat_cols if c != "_split"]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Handle unseen values by treating NaN as a separate category
    combined[col] = combined[col].fillna("Unknown")
    combined[col] = le.fit_transform(combined[col].astype(str))
    encoders[col] = le

logger.info(f"Label-encoded {len(cat_cols)} categorical columns")

# Impute remaining missing values

In [ ]:
from sklearn.impute import SimpleImputer

# Separate _split before imputation
split_col = combined["_split"].copy()
combined.drop(columns=["_split"], inplace=True)

# Numeric: median imputation (robust to outliers)
num_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(strategy="median")
combined[num_cols] = imputer.fit_transform(combined[num_cols])

combined["_split"] = split_col
logger.info(f"Imputed {len(num_cols)} numeric columns with median strategy")

# Split back & save

In [ ]:
train_final = combined[combined["_split"] == "train"].drop(columns=["_split"]).reset_index(drop=True)
test_final  = combined[combined["_split"] == "test"].drop(columns=["_split"]).reset_index(drop=True)

# Re-attach target
train_final["TARGET"] = y.values

# Save
train_final.to_parquet(DATA_PROC / "train_features.parquet", index=False)
test_final.to_parquet(DATA_PROC / "test_features.parquet",  index=False)

logger.info(f"Saved train: {train_final.shape} | test: {test_final.shape}")
print(f"\nFinal feature count : {train_final.shape[1] - 1}")
print(f"New engineered cols : {train_final.shape[1] - 1 - 122}")
print(f"Sample engineered features:")
print(train_final[["CREDIT_INCOME_RATIO","EXT_SOURCE_MEAN","EXT_SOURCE_DISAGREEMENT","AGE_YEARS"]].describe().round(3))